In [1]:
#| hide
import allos as al
import pandas as pd
import scanpy as sc
import urllib
import urllib.request
import gzip
import shutil
from pathlib import Path
from allos.readers_tests import *


# allos

> Storage and visualization of long read based single cell datasets
Allos consists from following modules:
- Palettes
- Transcript plots
- Transcript data
- Readers and tests
- AnndataIso
- Preprocessing
- Swithch Search
- Gene report


![Allos_logo](/data/analysis/data_mcandrew/00_allos_dev/allos/nbs/logo_allos.png)

## Install

```sh
pip install allos
```

# Basic workflow

The test dataset represents a data prepared with 10X genomics and sequenced on PromethION (Oford Nanopoere sequncing). The transcript and genes were quantified using SiCeLoRe (Single Cell Long Read) 2.1. The data are available at NCBI as two count matrices devided into [951 cells](https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3748nnn/GSM3748089/suppl/GSM3748089%5F951c.isoforms.matrix.txt.gz)
 and [190 cells](https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3748nnn/GSM3748087/suppl/GSM3748087%5F190c.isoforms.matrix.txt.gz)


1. Download the data and create an anndata object:

In [2]:
cell_951 = "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3748nnn/GSM3748089/suppl/GSM3748089%5F951c.isoforms.matrix.txt.gz"
cell_190 = "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3748nnn/GSM3748087/suppl/GSM3748087%5F190c.isoforms.matrix.txt.gz"


In [3]:
file_path = ("../resources/e18.mouse.clusters.csv")
df = pd.read_csv(file_path)
df['barcode'] = df.index.str.split('_').str[1]


mouse_data_str_1 = download_test_data(cell_190, output_filename='mouse_1.txt')
print("Test data downloaded successfully")


mouse_data_str_2 = download_test_data(cell_951, output_filename='mouse_2.txt')
print("Test data downloaded successfully")


mouse_1 = read_sicelore_isomatrix(file_path=mouse_data_str_1)
mouse_2 = read_sicelore_isomatrix(file_path=mouse_data_str_2)

combined_mouse_data = iso_concat([mouse_1, mouse_2], batch_type='numeric')


combined_mouse_data.obs_names_make_unique()
# Step 1: Remove any duplicate barcodes in the DataFrame
df_unique = df.drop_duplicates(subset='barcode')

# Step 2: Filter the DataFrame to include only the barcodes present in the AnnData object
df_filtered = df_unique[df_unique['barcode'].isin(combined_mouse_data.obs_names)]

# Step 3: Set the index of the filtered DataFrame to 'barcode' to make the merge easier
df_filtered.set_index('barcode', inplace=True)

# Step 4: Create a DataFrame from the obs DataFrame of the AnnData object to ensure the same index
obs_df = combined_mouse_data.obs.copy()

# Step 5: Initialize a new column 'cell_type' with NaN values in the obs DataFrame
obs_df['cell_type'] = pd.NA

# Step 6: Update the 'cell_type' column with values from the filtered DataFrame where indices match
obs_df.update(df_filtered['illumina.ident'].rename('cell_type'))

# Step 7: Ensure the index is unique and assign the updated DataFrame back to the obs attribute of the AnnData object
if obs_df.index.is_unique:
    combined_mouse_data.obs = obs_df
else:
    raise ValueError("The index of the obs DataFrame is not unique.")

# Now, the 'cell_type' column should be added to the obs DataFrame of your AnnData object
combined_mouse_data = combined_mouse_data[~combined_mouse_data.obs['cell_type'].isna()]

✅ File already exists at: /home/diamant/.conda/envs/iso_swt/lib/python3.9/site-packages/allos/resources/data/mouse_1.txt
Test data downloaded successfully
✅ File already exists at: /home/diamant/.conda/envs/iso_swt/lib/python3.9/site-packages/allos/resources/data/mouse_2.txt
Test data downloaded successfully
Error reading file at /home/diamant/.conda/envs/iso_swt/lib/python3.9/site-packages/allos/resources/data/mouse_1.txt: 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte

## Data preprocessing

Ensure that the dataset contains the 'cell_type' and 'barcodes' columns in observation (adata.obs) and make logtransformatin before running the SwitchSearch test.

In [6]:
sc.pp.log1p(combined_mouse_data)

/home/diamant/.conda/envs/iso_swt/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:373: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


## Switch Search 
Switch_search provides two statistical methods to find the swithching isoforms. The default method is set to "isoSwitch", to choose Dirichlet model set method="Dirichlet"

In [7]:
from allos.switch_search import SwitchSearch
combined_mouse_data = SwitchSearch(combined_mouse_data)
switch_list = combined_mouse_data.find_switching_isoforms(cell_group_column="cell_type")

Sort the values by adjusted p-value and choose the transcipts of interest:

In [8]:
switch_list.sort_values(by=["pvals_adj", "logfoldchanges"])

,names,scores,logfoldchanges,pvals,pvals_adj,group_1,group_2,contrast,geneId,n_cells_group_1,n_cells_group_2,total_cells,adj_pval,direction,percent_expressed_group_1,percent_expressed_group_2
9249,ENSMUST00000034834.15,11.252645,2.955992,2.247471e-29,1.671805e-26,cycling radial glia,mature Glutamatergic,cycling radial glia__mature Glutamatergic,Pkm,117,275,392,2.844541e-24,2.955992,88.888889,42.181818
2293,ENSMUST00000107849.9,11.248262,2.711614,2.362000e-29,1.717066e-26,mature Glutamatergic,cycling radial glia,cycling radial glia__mature Glutamatergic,Clta,275,117,392,2.905632e-24,-2.711614,87.636364,39.316239
9250,ENSMUST00000163694.3,11.143052,2.630076,7.742013e-29,5.503023e-26,mature Glutamatergic,cycling radial glia,cycling radial glia__mature Glutamatergic,Pkm,275,117,392,8.785597e-24,-2.630076,89.818182,43.589744
9274,ENSMUST00000034834.15,10.952438,3.768384,6.468268e-28,8.995392e-25,radial glia,mature Glutamatergic,mature Glutamatergic__radial glia,Pkm,68,275,343,1.178586e-22,-3.768384,95.588235,42.181818
9273,ENSMUST00000163694.3,10.737323,4.453093,6.798722e-27,8.363997e-24,mature Glutamatergic,radial glia,mature Glutamatergic__radial glia,Pkm,275,68,343,9.444321e-22,4.453093,89.818182,16.176471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6847,ENSMUST00000106171.8,3.723201,1.599538,1.967130e-04,7.758400e-03,mature Glutamatergic,radial glia,mature Glutamatergic__radial glia,Meaf6,275,68,343,3.162429e-02,1.599538,48.363636,22.058824
6848,ENSMUST00000154689.7,3.712274,6.692658,2.054053e-04,8.022091e-03,radial glia,mature Glutamatergic,mature Glutamatergic__radial glia,Meaf6,68,275,343,3.260497e-02,-6.692658,29.411765,0.363636
7659,ENSMUST00000218127.1,3.677007,0.916295,2.359862e-04,8.616730e-03,cycling radial glia,mature Glutamatergic,cycling radial glia__mature Glutamatergic,Myl6,117,275,392,3.479863e-02,0.916295,58.974359,37.818182
15177,ENSMUST00000118830.7,3.628299,1.390801,2.852949e-04,1.023032e-02,cycling radial glia,mature Glutamatergic,cycling radial glia__mature Glutamatergic,Vps29,117,275,392,4.070367e-02,1.390801,41.025641,20.000000


In [10]:
from allos.transcript_data import TranscriptData

In [ ]:
# import os
# import urllib.request
# from pathlib import Path

# # Example Ensembl URLs for mouse GRCm39 (release 109)
# gtf_url = "ftp://ftp.ensembl.org/pub/release-109/gtf/mus_musculus/Mus_musculus.GRCm39.109.gtf.gz"
# fasta_url = "ftp://ftp.ensembl.org/pub/release-109/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz"

# # Store data one directory back
# data_dir = Path("..") / "data"
# data_dir.mkdir(parents=True, exist_ok=True)

# gtf_file_local = data_dir / "Mus_musculus.GRCm39.109.gtf.gz"
# fasta_file_local = data_dir / "Mus_musculus.GRCm39.dna.primary_assembly.fa.gz"

# # Download if not already present
# if not gtf_file_local.is_file():
#     print(f"Downloading {gtf_url}...")
#     urllib.request.urlretrieve(gtf_url, gtf_file_local)

# if not fasta_file_local.is_file():
#     print(f"Downloading {fasta_url}...")
#     urllib.request.urlretrieve(fasta_url, fasta_file_local)

# # Instantiate your TranscriptData
# td = TranscriptData(
#     gtf_file=gtf_file_local,
#     reference_fasta=fasta_file_local
# )

# # Now you can make queries like:
# example_transcript_id = "ENSMUST00000070533"  # e.g., for mouse
# exons = td.get_exons(example_transcript_id)
# print("Exons:", exons)

Exons: +--------------+----------------+------------+-----------+-------+
|   Chromosome | Source         | Feature    |     Start | +22   |
|   (category) | (object)       | (object)   |   (int64) | ...   |
|--------------+----------------+------------+-----------+-------|
|            1 | ensembl_havana | exon       |   3740774 | ...   |
|            1 | ensembl_havana | exon       |   3491924 | ...   |
|            1 | ensembl_havana | exon       |   3284704 | ...   |
+--------------+----------------+------------+-----------+-------+
Stranded PyRanges object has 3 rows and 26 columns from 1 chromosomes.
For printing, the PyRanges was sorted on Chromosome and Strand.
22 hidden columns: End, Score, Strand, Frame, gene_id, gene_version, ... (+ 16 more.)


### Plot transcript module
Plot transcript module can be used on its own to explore the structure on known or new transcripts. 
Transcript's structure can be visualized with **draw_transcript** function from custom coordiantes or directly by indicating a valid Ensemble id. 

 


In [12]:
from allos.transcript_plots import TranscriptPlots


Multiple transcripts can be vizualized on one panel with the function draw_transcripts_list where the transcripts' ids are provided as a list. The list can be a mix of known and novev, custom defined transcripts:


In [14]:
td = TranscriptData(gtf_file_local)

In [15]:
tp = TranscriptPlots(gtf_file_local, fasta_file_local)

NameError: name 'TranscriptData' is not defined

In [ ]:
tp.draw_transcripts_list(["ENSMUST00000107851", "ENSMUST00000107846", "ENSMUST00000107847"], colors=ghibli)

get_transcript_info helps to retrieve information about the transcripts of interest:

In [ ]:
import pandas as pd
transcripts = [
    tp.get_transcript_info("ENSMUST00000107851"),
    tp.get_transcript_info("ENSMUST00000107846"),
    tp.get_transcript_info("ENSMUST00000107847")
]
pd.DataFrame(transcripts)

### Gene report module
Offers a complete pipeline to discover and visualize differentially expressed and switching isoforms among different cell types and conditions in long reads based single RNA-seq data. Allos supports several technologies and tools producing long reads based scRNA seq datasets:
Oxford Nanopore, PacBio, Smartseq2 and such tools as Sicelore, Isocsceles, Kallisto etc. The output from other tools can be adapted to a universal reader.

In [20]:
import allos.gene_report as gr

<Figure size 100x100 with 0 Axes>

## 1. Initialization
The AnnDataIso object initializes with:

Input: AnnData object and an optional cell types DataFrame.
### Setup:
Filters genes with multiple isoforms.
Calculates isoform percentages per gene for each cell type.
Prepares the data for downstream analyses.
### Usage:

In [21]:
path = al.readers_tests.download_test_data()

Starting download of test data from https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3748nnn/GSM3748087/suppl/GSM3748087%5F190c.isoforms.matrix.txt.gz
File downloaded successfully
File extracted successfully


In [22]:
adata_iso = al.switch_search.create_iso_adata(path)

In [23]:
adata_iso = al.anndata_iso.AnnDataIso(adata_iso)


4. Data Visualization
a. Isoform Summary
Method: plot_isoforms_summary()
Purpose: Summarizes isoform switching and frequencies across genes.
Subplots:
Bar plot: Percentage of genes with multiple isoforms.
Bar plot: Frequency of isoforms per gene.
Boxplot: Number of genes expressed per cell type.
Usage:



In [24]:
#| hide 
from nbdev.showdoc import *


In [25]:
#| hide
import nbdev; nbdev.nbdev_export()
adata_iso.plot_isoforms_summary()

AttributeError: 'AnnDataIso' object has no attribute 'plot_isoforms_summary'

### Gene-Specific Transcripts
Method: plot_transcripts_per_cell_type()
Visualizes isoform expression across cell types for a specific gene.
Parameters:
gene_name: The gene to visualize.
trs_to_show: List of transcript IDs to include (optional).
Usage:

In [ ]:
adata_iso.plot_transcripts_per_cell_type("GeneName")

### Gene Summary
Method: draw_gene_summary()
Purpose: Generates a comprehensive visualization of transcript counts, per-cell-type breakdown, and transcripts structures for a specific gene.
Parameters:
gene_name: Target gene for visualization.
trs_to_show: List of transcripts to highlight.
Usage:

In [ ]:
adata_iso.draw_gene_summary("GeneName")

## Statistical Analysis
a. Find Switching Isoforms
Method: find_switching_isoforms()
Purpose: Identifies genes with significant isoform switching between cell types using statistical tests.
Parameters:
cell_group_column: Column specifying cell group labels.
min_count: Minimum expression count threshold.
min_diff: Minimum expression difference for isoform detection.
Usage:

In [ ]:
switching_genes = adata_iso.find_switching_isoforms(cell_group_column="cell_type")

## Likelihood Ratio Test
Method: __compare_groups()
Purpose: Tests for significant isoform usage differences between two groups.
Parameters:
group_1_label and group_2_label: Names of the groups to compare.
cell_group_column: Column specifying cell group labels.
gene_id: Gene ID to test.

In [ ]:
adata_iso.plot_isoforms_summary()

### Gene-Specific Transcripts
Method: plot_transcripts_per_cell_type()
Visualizes isoform expression across cell types for a specific gene.
Parameters:
gene_name: The gene to visualize.
trs_to_show: List of transcript IDs to include (optional).
Usage:

In [ ]:
adata_iso.plot_transcripts_per_cell_type("GeneName")

### Gene Summary
Method: draw_gene_summary()
Purpose: Generates a comprehensive visualization of transcript counts, per-cell-type breakdown, and transcripts structures for a specific gene.
Parameters:
gene_name: Target gene for visualization.
trs_to_show: List of transcripts to highlight.
Usage:

In [ ]:
adata_iso.draw_gene_summary("GeneName")

## Statistical Analysis
a. Find Switching Isoforms
Method: find_switching_isoforms()
Purpose: Identifies genes with significant isoform switching between cell types using statistical tests.
Parameters:
cell_group_column: Column specifying cell group labels.
min_count: Minimum expression count threshold.
min_diff: Minimum expression difference for isoform detection.
Usage:

In [ ]:
switching_genes = adata_iso.find_switching_isoforms(cell_group_column="cell_type")

## Likelihood Ratio Test
Method: __compare_groups()
Purpose: Tests for significant isoform usage differences between two groups.
Parameters:
group_1_label and group_2_label: Names of the groups to compare.
cell_group_column: Column specifying cell group labels.
gene_id: Gene ID to test.

In [ ]:
adata_iso.plot_isoforms_summary()

### Gene-Specific Transcripts
Method: plot_transcripts_per_cell_type()
Visualizes isoform expression across cell types for a specific gene.
Parameters:
gene_name: The gene to visualize.
trs_to_show: List of transcript IDs to include (optional).
Usage:

In [ ]:
adata_iso.plot_transcripts_per_cell_type("GeneName")

### Gene Summary
Method: draw_gene_summary()
Purpose: Generates a comprehensive visualization of transcript counts, per-cell-type breakdown, and transcripts structures for a specific gene.
Parameters:
gene_name: Target gene for visualization.
trs_to_show: List of transcripts to highlight.
Usage:

In [ ]:
adata_iso.draw_gene_summary("GeneName")

## Statistical Analysis
a. Find Switching Isoforms
Method: find_switching_isoforms()
Purpose: Identifies genes with significant isoform switching between cell types using statistical tests.
Parameters:
cell_group_column: Column specifying cell group labels.
min_count: Minimum expression count threshold.
min_diff: Minimum expression difference for isoform detection.
Usage:

In [ ]:
switching_genes = adata_iso.find_switching_isoforms(cell_group_column="cell_type")

## Likelihood Ratio Test
Method: __compare_groups()
Purpose: Tests for significant isoform usage differences between two groups.
Parameters:
group_1_label and group_2_label: Names of the groups to compare.
cell_group_column: Column specifying cell group labels.
gene_id: Gene ID to test.

## Isoform Rating
Major vs. Minor Isoforms
Method: find_major_minor_isoforms()
Purpose: Identifies major and minor isoforms for each gene across cell types.
Usage:



In [ ]:
major_isoforms = adata_iso.find_major_minor_isoforms()